# Causal gMLP Experiments
COMP6242 RNN's Revenge — Causal gMLP with windowed SGU (w=256)

Runs 9 experiments: 3 tasks × 3 lengths.

| Task | Lengths | Steps | Notes |
|------|---------|-------|-------|
| Shakespeare | 256 / 1024 / 2048 | 5K | |
| Copy | short / medium / long | 5K | |
| Induction | short / medium / long | 20K | lr_decay_steps=50000 |

Protocol: dropout 0.05, AdamW (0.9/0.95), wd=0.1, lr 3e-4→3e-5, seed 42.

Note: gMLP's 256-token spatial window means it cannot see the first pattern
occurrence at long sequence lengths (T=2048). Induction accuracy degrades
sharply for the 'long' tier — known architectural limitation.

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'bf16: {torch.cuda.is_bf16_supported()}')

In [ ]:
%cd /content/DL-RNNs-Revenge/vibhansh-gMLP

## Data Setup
Symlink to shared data generated by the transformer notebook (or generate fresh).

In [ ]:
import os
if not os.path.exists('data'):
    if os.path.exists('/content/DL-RNNs-Revenge/transformer_project/data'):
        os.symlink('/content/DL-RNNs-Revenge/transformer_project/data', 'data')
    else:
        os.chdir('/content/DL-RNNs-Revenge/transformer_project')
        os.system('python data/tinyshakespeare/prepare.py')
        os.system('python generate_longrange_copy.py')
        os.system('python generate_induction.py')
        os.chdir('/content/DL-RNNs-Revenge/vibhansh-gMLP')
        os.symlink('/content/DL-RNNs-Revenge/transformer_project/data', 'data')
print('Data contents:', os.listdir('data'))

## Shakespeare × 3

In [ ]:
!python train.py --task shakespeare --block_size 256 --max_steps 5000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_path data/tinyshakespeare/input.txt \
    --eval_interval 250 --output_dir results

!python train.py --task shakespeare --block_size 1024 --max_steps 5000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_path data/tinyshakespeare/input.txt \
    --eval_interval 250 --output_dir results

!python train.py --task shakespeare --block_size 2048 --max_steps 5000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_path data/tinyshakespeare/input.txt \
    --eval_interval 250 --output_dir results

## Long-range Copy × 3

In [ ]:
!python train.py --task copy --length short --max_steps 5000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_dir data --eval_interval 250 --output_dir results

!python train.py --task copy --length medium --max_steps 5000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_dir data --eval_interval 250 --output_dir results

!python train.py --task copy --length long --max_steps 5000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_dir data --eval_interval 250 --output_dir results

## Induction × 3 (20K steps, lr_decay 50K)

In [ ]:
!python train.py --task induction --length short --max_steps 20000 --lr_decay_steps 50000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_dir data --eval_interval 1000 --output_dir results

!python train.py --task induction --length medium --max_steps 20000 --lr_decay_steps 50000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_dir data --eval_interval 1000 --output_dir results

!python train.py --task induction --length long --max_steps 20000 --lr_decay_steps 50000 \
    --dropout 0.05 --batch_size 64 --learning_rate 3e-4 --weight_decay 0.1 \
    --warmup_steps 200 --grad_clip 1.0 --seed 42 \
    --data_dir data --eval_interval 1000 --output_dir results

## Collect Results

In [ ]:
import json, glob

print(f"{'File':<45} {'Val PPL':>10} {'Disc PPL':>10} {'Acc':>8}")
print('-' * 75)
for p in sorted(glob.glob('results/gmlp_*.json')):
    with open(p) as f:
        s = json.load(f)
    name = p.split('/')[-1]
    ppl = s['best_val_ppl']
    fr = s.get('final_results', {})
    disc = fr.get('discriminating_ppl', '-')
    acc = fr.get('induction5_accuracy', fr.get('accuracy', '-'))
    disc_str = f"{disc:.4f}" if isinstance(disc, float) else '-'
    acc_str = f"{acc:.4f}" if isinstance(acc, float) else '-'
    print(f"{name:<45} {ppl:>10.4f} {disc_str:>10} {acc_str:>8}")